In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from dtw import dtw

In [ ]:
def row_mean_ci(row):
    mean = row.mean()
    sem = stats.sem(row, nan_policy='omit')
    ci_low, ci_high = stats.t.interval(
        0.95, 
        len(row.dropna()) - 1,
        loc=mean, 
        scale=sem
    )
    return pd.Series({'mean_scaled': mean, 'ci_low_scaled': ci_low, 'ci_high_scaled': ci_high})

def plot_exacttot_expr_mean_ci(person_df, counts_df, gene_cols, subject_id, ax=None):
    show_legends = False
    if ax is None:
        show_legends = True
        fig, ax = plt.subplots(figsize=(10, 4), dpi=600)
    else:
        fig = ax.figure
        show_legends = False
        
    # min max scaling to put on gene expression on same scale as exac scores
    exac_min, exac_max = person_df['exacttot'].min(), person_df['exacttot'].max()
    
    # add an offset so that its not 0/0
    eps = 1e-8
    scaled_df = counts_df.copy()
    # scale each gene to the scale of exact scores
    for g in gene_cols:
        g_min, g_max = counts_df[g].min(), counts_df[g].max()
        scaled_df[g] = (counts_df[g] - (g_min - eps)) / ((g_max + eps) - (g_min - eps)) * (exac_max - exac_min) + exac_min
        
    # calculate mean + CI
    scaled_stats = scaled_df[gene_cols].apply(row_mean_ci, axis=1)
    scaled_df = pd.concat([scaled_df, scaled_stats], axis=1)

    exac = person_df[person_df['exacerbation_flag'] == 1]
    antibiotics = person_df[person_df['antibiotics_y/n'] == 1]
    steroids = person_df[person_df['steroids_y/n'] == 1]

    baseline = person_df['baseline_exacttot'].iloc[0]
    week_day_map = person_df.groupby('week_num')['dummy_days'].min()
    xticks = week_day_map.values

    # subject_id = person_df['SID'].iloc[0]

    ax.axhline(y=baseline, color='black', linestyle='--', label='baseline EXACT')

    # ax.scatter(exac['dummy_days'], exac['exacttot'], color='orange',
    #            label='exacerbation event', zorder=3, s=60)
    # ax.scatter(antibiotics['dummy_days'], antibiotics['exacttot'], color='purple',
            #    label='antibiotics', zorder=4, marker='v', s=50)
    # ax.scatter(steroids['dummy_days'], steroids['exacttot'], color='#009E73',
            #    label='steroids', zorder=4, marker='^', s=50)

    ax.plot(person_df['dummy_days'], person_df['exacttot'],
            color='gray', marker='o', label='EXACT score')

    ax.plot(scaled_df['dummy_days'], scaled_df['mean_scaled'],
            linestyle='--', marker='x', label='mean expression')

    ax.fill_between(scaled_df['dummy_days'], scaled_df['ci_low_scaled'], scaled_df['ci_high_scaled'],
                     color='blue', alpha=0.2, label='95% CI')

    ax.set_title(f'Subject {subject_id}')
    ax.set_xlabel('Study Week')
    ax.set_ylabel('Score')
    ax.grid(True)
    ax.set_xticks(xticks)
    ax.set_xticklabels(week_day_map.index.astype(str))
    ax.set_xlim(xticks[0], xticks[-1] + 7)

    if show_legends:
        ax.legend()
        
    return fig, ax


In [ ]:
df = pd.read_csv('../data/periCOPD_exac_pheno_clean.csv')

df['study_date'] = pd.to_datetime(df['study_date'])
df['weekday_num'] = df['study_date'].dt.weekday

In [ ]:
missing_dates = []

for sid, g in df.groupby('SID'):
    dates = pd.to_datetime(sorted(g['study_date'].dropna().unique()))
    if len(dates) == 0:
        continue

    first_date = dates[0]
    last_date = dates[-1]
    full_range = pd.date_range(first_date, last_date, freq='D')
    present = set(dates)

    for d in full_range:
        if d in present:
            continue
        days_from_start = (d - first_date).days
        missing_dates.append({
            'SID': sid,
            'missing_date': d,
            'days_from_start': days_from_start
        })

missing_df = pd.DataFrame(missing_dates)
missing_df

In [ ]:
summary = df.groupby('SID')['study_date'].agg(
    n_observed='nunique',
    first_date='min',
    last_date='max'
).reset_index()

summary['window_days'] = (summary['last_date'] - summary['first_date']).dt.days + 1
summary['n_missing_in_window'] = summary['window_days'] - summary['n_observed']

summary['pct_complete'] = (1 - summary['n_missing_in_window'] / summary['window_days']) * 100
summary['pct_complete'] = summary['pct_complete'].round(1)
summary

In [ ]:
pooled = (1 - summary['n_missing_in_window'].sum() / summary['window_days'].sum()) * 100
mean_pct = summary['pct_complete'].mean()
mean_pct

In [ ]:
# make a dummy day so that the week intervals are equal
week_min_day = df.groupby(['SID', 'week_num'])['days'].transform('min')
df['day_in_week'] = df['days'] - week_min_day
sid_week = df[['SID', 'week_num']].drop_duplicates()
days7 = pd.DataFrame({'day_in_week': np.arange(7)})
grid = sid_week.merge(days7, how='cross')
df = grid.merge(df, on=['SID', 'week_num', 'day_in_week'], how='left')
df['dummy_days'] = df['week_num'] * 7 + df['day_in_week']

In [ ]:
counts_subset = pd.read_csv('../data/WGCNA_AMEND_impulse_short_in_blue_genes_norm.csv')
counts_subset = counts_subset.rename(columns={'subject_id': 'SID'})

In [ ]:
# each friday
collation_days = (df[df['weekday_num'] == 4][['SID', 'week_num', 'days', 'dummy_days']])
# make the first one day 0 in dummy days
collation_days0 = pd.DataFrame({
    'SID': ['S1001', 'S1002', 'S1003', 'S1004'],
    'week_num': 0, 'days': 0, 'dummy_days': 0})
collation_days = pd.concat([collation_days0, collation_days], ignore_index=True)
collation_days = collation_days.sort_values(['SID', 'week_num']).reset_index(drop=True)

counts_subset = counts_subset.merge(collation_days, on=['SID', 'week_num'])

gene_cols = [col for col in counts_subset.columns if col not in ['SID', 'week_num', 'days', 'dummy_days']]

In [ ]:
df_1 = df[df['SID'] == 'S1001']
df_1 = df_1.dropna(subset=['exacttot'])
counts_1_subset = counts_subset[counts_subset['SID'] == 'S1001'].copy()

df_2 = df[df['SID'] == 'S1002']
df_2 = df_2.dropna(subset=['exacttot'])
counts_2_subset = counts_subset[counts_subset['SID'] == 'S1002'].copy()

df_3 = df[df['SID'] == 'S1003'].copy()
# no pheno data for week 10
df_3.loc[df_3['week_num'] >= 11, 'dummy_days'] -= 7
df_3 = df_3.dropna(subset=['exacttot'])

counts_3_subset = counts_subset[counts_subset['SID'] == 'S1003'].copy()
counts_3_subset.loc[counts_3_subset['week_num'] >= 11, 'dummy_days'] -= 7

df_4 = df[df['SID'] == 'S1004']
df_4 = df_4.dropna(subset=['exacttot'])
counts_4_subset = counts_subset[counts_subset['SID'] == 'S1004'].copy()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 8), dpi=300)
axes = axes.flatten()

datasets = [
    (df_1, counts_1_subset),
    (df_2, counts_2_subset),
    (df_3, counts_3_subset),
    (df_4, counts_4_subset)
]
subject_ids = [1,2,3,4]

for i, ((df_sub, counts_sub), subject_id) in enumerate(zip(datasets, subject_ids)):
    plot_exacttot_expr_mean_ci(df_sub, counts_sub, gene_cols, subject_id, ax=axes[i])

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.03), ncol=5, fontsize=12)
# fig.suptitle('EXACT & Mean Module Expression (95% CI) Across 4 Subjects', fontsize=16, y=0.95)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()
# ChanRev40
fig.savefig("../figures_final/pheno_module_mean_4subjects_no_prescription.png", dpi=600, bbox_inches="tight")